# 01 – Business Understanding

**Project:** Diabetes Prediction ML  
**Seminar:** Advanced Applied Data Science – Goethe University Frankfurt  
**Dataset:** CDC BRFSS 2015 – Diabetes Health Indicators (UCI #891)  
**Target:** `Diabetes_binary` (0 = No Diabetes, 1 = Prediabetes/Diabetes)  
**CRISP-DM phase:** 1 – Business Understanding

---

## Purpose of this notebook

Before any data exploration or modelling, we clarify **what** we actually want to predict, **for whom**, **how** the result is used, and **which metrics** are appropriate. All subsequent notebooks (EDA, preprocessing, modelling, evaluation) build on these decisions.

## 1. Use Case

**Risk stratification / population screening.**

A person answers a few web-based questions about their health status and lifestyle (BMI, age, known high blood pressure, physical activity, general well-being, …). Based on these answers, the tool produces a **binary risk assessment** for prediabetes/diabetes: for an elevated-risk profile it recommends seeking medical clarification, otherwise no action. The model does not replace a clinical diagnosis. 

Diabetes is also the leading driver of severe complications such as end-stage renal disease — in 2014, roughly 44 % of new ESRD cases in the U.S. and Puerto Rico were attributed to diabetes (Burrows et al., 2017) — which underscores the value of early, low-barrier risk identification.

**Why this use case fits the dataset:**
- BRFSS is itself a self-reported telephone survey → training and deployment distributions are identical.
- No clinical biomarkers required (no HbA1c, no glucose).
- A low-barrier recommendation ("see a doctor for a blood test" or "adjust lifestyle") is low-cost and low-risk.

**On `Stroke` and `HeartDiseaseorAttack`:** Both variables are recorded in BRFSS as self-reported diagnoses — they are observable comorbidity indicators, not causal early-prevention variables. Their presence in the feature set shifts the model's interpretation from pure early detection toward general risk assessment of existing high-risk profiles. This is methodologically acceptable in the BRFSS screening context.

## 2. Stakeholders

- **Primary:** end users (individuals who want to assess their own risk).
- **Secondary:** public-health context (population screening, awareness).

## 3. Cost Matrix

| Prediction \ Truth | Diabetes/Prediabetes | No diabetes |
|---|---|---|
| **Model: elevated risk** | TP: person seeks medical clarification → possible diagnosis and early intervention. **High benefit.** | FP: doctor visit without finding. **Low cost.** For a high-risk profile, even useful prevention. |
| **Model: low risk** | FN: person remains undetected. **High cost** — missed early intervention and late complications such as end-stage renal disease (Burrows et al., 2017). | TN: correctly cleared. |

**Consequence:** FN ≫ FP → **recall priority.** We accept lower precision in favour of high recall.

## 4. Label Noise: a central methodological insight

### 4.1 How the target is recorded in the dataset

The target variable is based on the BRFSS 2015 question "**(Ever told) you have diabetes**" with the prologue "**Has a doctor, nurse, or other health professional ever told you that you had any of the following?**" — i.e. purely **self-reported information about a medical diagnosis**, not the presence of diabetes itself. Women affected only during pregnancy, as well as respondents answering "don't know"/"refused", were counted as non-diabetic in the UCI variant; "Yes" and "pre-diabetes/borderline" were merged into `Diabetes_binary = 1`.

Source: [BRFSS 2015 Codebook, variable `DIABETE3`](https://www.cdc.gov/brfss/annual_data/2015/pdf/codebook15_llcp.pdf).

→ The label therefore encodes **diagnosis status**, not necessarily **disease status**. Negative labels may contain undiagnosed cases. This is a plausible methodological limitation based on known under-diagnosis rates — not a fact directly observed in the dataset.

### 4.2 Share of undiagnosed cases

A substantial fraction of the U.S. population is unaware that they have diabetes or prediabetes:

- **Diabetes:** ~27.6 % of adults with diabetes in the U.S. are undiagnosed (≈ 11 million people, 2023). Source: [CDC National Diabetes Statistics Report](https://www.cdc.gov/diabetes/php/data-research/index.html), January 2026.
- **Prediabetes:** ~80 % of adults with prediabetes do not know they have it. Source: [CDC – A U.S. Report Card on Diabetes](https://www.cdc.gov/diabetes/communication-resources/diabetes-statistics.html).
- **Validation via an independent data source:** in NHANES 2021–2023 the undiagnosed-diabetes prevalence among adults aged 20+ is 4.2 % at a total prevalence of 14.3 % → ~29 % of the diabetic population undiagnosed. Source: [NCHS Data Brief 516](https://www.cdc.gov/nchs/products/databriefs/db516.htm), November 2024.

### 4.3 Consequences for our model

- Many "negatives" in the dataset may in reality be positive (asymmetric label noise on the negative class — a plausible assumption, not directly observed).
- The model may be penalised for correct predictions when it recognises undiagnosed high-risk profiles.
- The *measured* PR-AUC may therefore *underestimate* the *true* model quality.

**To show in the report:** false-positive profile analysis — do our FPs look like an undiagnosed high-risk population (high BMI, HighBP, older age, poor GenHlth)? Some of the FPs may stem from undiagnosed or label-unrecorded high-risk cases — that would point to systematic label noise rather than model error.

## 5. Metrics — Selection and Justification

This section explains in detail **which metrics we use and why**. The choice is not obvious: on a strongly imbalanced dataset like this one (~14 % positive class) the common standard metrics give misleading results. We derive our decision step by step.

### 5.1 Foundation: the confusion matrix

All classification metrics are built on four numbers. For binary classification:

|   | Model: positive | Model: negative |
|---|---|---|
| **Actually positive** | **TP** – correctly detected | **FN** – missed |
| **Actually negative** | **FP** – false alarm | **TN** – correctly cleared |

In medical screening, **FN is especially costly**: someone with diabetes is sent home with no follow-up. **FP is usually cheaper**: someone gets an unnecessary second check, but no lasting harm. → see the cost matrix above.

### 5.2 The individual metrics — each answers a concrete question

**Accuracy = (TP + TN) / total**
- Question: *"How often is the model right overall?"*
- **Problem under imbalance:** a model that always says "no diabetes" reaches ~86 % accuracy here and is clinically worthless — every diabetic is missed.
- **→ Accuracy is *not* used as the primary metric.**

**Precision = TP / (TP + FP)**
- Question: *"When the model raises an alarm, how often is it right?"*
- High = few false alarms.
- Depends on the chosen threshold.

**Recall (sensitivity) = TP / (TP + FN)**
- Question: *"How many of the true diabetics do I catch?"*
- High = few missed cases.
- **Almost always the most important quantity in medical screening.**
- Depends on the chosen threshold.

**Specificity = TN / (TN + FP)**
- Question: *"How many of the healthy do I correctly leave alone?"*
- The counterpart to recall, focused on the negative class.

**F1 = 2 · (Precision · Recall) / (Precision + Recall)**
- Harmonic mean of precision and recall.
- Becomes small as soon as either quantity is small.
- **Drawback:** depends on the chosen threshold → says little about the model "as a whole".

**MCC (Matthews Correlation Coefficient)**
- A single-number summary that uses all four cells of the confusion matrix; it is only high when the model does well on **both** classes, which makes it more informative than accuracy or F1 under imbalance (Chicco & Jurman, 2020).
- **Two caveats for our use case:** it is threshold-dependent (an operating-point metric, not a model-selection metric), and it is **symmetric** — it weights FP and FN equally, whereas our cost structure is deliberately FN ≫ FP. We therefore report it as a balanced summary but do not use it as the deciding criterion (see 5.8).

### 5.3 Key insight: precision and recall depend on the threshold

A classification model does not really output a 0/1 prediction but a **probability between 0 and 1**. We ourselves decide from which threshold we call a case "positive".

- Threshold = 0.5 (default): moderate precision, moderate recall.
- Threshold = 0.2: more people are flagged as positive → **recall rises, precision falls**.
- Threshold = 0.8: only very confident cases are flagged → **precision rises, recall falls**.

So there is not *one* precision and *one* recall — there is a **whole curve**. A sensible primary metric should therefore be **threshold-independent** and assess model quality across all possible thresholds.

### 5.4 ROC-AUC vs. PR-AUC — what the curves plot

Both curves are produced by lowering the threshold slowly from 1.0 to 0.0; each threshold gives one point on the curve, defined by two numbers:

- **ROC curve:** recall (= TPR) against false-positive rate (FPR = FP / (FP + TN))
- **PR curve:** precision against recall

The area under each curve is the "AUC". The decisive difference lies in **which counts sit in the denominator**:

- Recall = TP / (TP + FN) → denominator = all *actual* positives (the real diabetics)
- FPR = FP / (FP + TN) → **denominator = all *actual* negatives** (huge under imbalance!)
- Precision = TP / (TP + FP) → denominator = all *predicted* positives (small)

The next section makes concrete why this denominator difference is what makes ROC-AUC look optimistic under imbalance.

### 5.5 A concrete example — why ROC-AUC is less informative under imbalance

Imagine 1,000 people: 140 true diabetics, 860 without diabetes (matching our class distribution).

The model flags 200 people as positive. Of these, 100 are truly diabetic (TP) and 100 are false alarms (FP):
- TP = 100, FN = 40, FP = 100, TN = 760

Computed metrics:

| Metric | Value | Interpretation |
|---|---|---|
| **Recall** | 100 / 140 = **0.71** | 71 % of diabetics caught |
| **Precision** | 100 / 200 = **0.50** | every second alarm is false |
| **FPR** | 100 / 860 = **0.12** | *looks small* |

**The FPR is only 0.12 — even though half of all alarms are false.**

Why? Because the denominator is 860. 100 errors against 860 negatives look minor. → **ROC-AUC is less sensitive to improvements on the positive class under imbalance**, because the large negative class dominates the FPR denominator.

Precision, by contrast, shows the problem directly: 0.50 clearly says "half are wrong". → **PR-AUC uses no TN count in its denominator and therefore stays focused on the minority class.**

**Comparison on balanced data** (500/500, same predictions):
- FPR = 100 / 500 = **0.20** instead of 0.12 → the problem would show more strongly in the ROC.
- Precision unchanged at 0.50.

**→ ROC-AUC is a valid measure but is dampened under strong imbalance by the large negative class. PR-AUC remains sensitive to improvements on the positive class even under imbalance.** This is the rationale formalised by Saito & Rehmsmeier (2015), who show that the precision–recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets.

### 5.6 Baselines compared

| Metric | Random baseline | Note |
|---|---|---|
| **ROC-AUC** | always 0.5 | independent of the class distribution |
| **PR-AUC** | = class prevalence | here **≈ 0.14** |

Consequence for interpretation:
- If someone reports "ROC-AUC = 0.80", at 14 % prevalence that only says the model separates *somehow*. It says little about whether precision is usable at a practically applicable threshold.
- **PR-AUC = 0.44 vs. baseline 0.14** says directly: ~3× better than chance, in exactly the discipline that matters here.

### 5.7 Resulting metric strategy

| Metric | Role | Justification |
|---|---|---|
| **PR-AUC (Average Precision)** | **Primary** (model selection) | Threshold-independent, focused on the minority class, robust to imbalance. No-skill baseline = 0.14. |
| ROC-AUC | Secondary (comparison) | Standard in the literature, enables benchmarking. Slightly optimistically biased under imbalance — not a sole decision criterion. |
| Confusion matrix + Precision/Recall/F1 @ threshold | **Mandatory** (binary output) | Standard reporting of the binary classifier at the threshold chosen in Section 6. |
| MCC @ threshold | Secondary (balanced summary) | Confusion-matrix-wide quality at the operating point; symmetric, so not the deciding criterion (see 5.8). |
| Subgroup performance (PR-AUC per group) | Secondary (fairness) | The tool must work comparably across sex, age, income. |
| Accuracy | **not** a primary metric | Misleading under imbalance (86 % from a trivial model). Context figure only. |
| F1 | only at the chosen threshold | Threshold-dependent, hence not for model selection. |

All threshold-dependent metrics (Precision, Recall, F1, MCC) are reported at the **chosen operating point** (Section 6), not at the default 0.5.

### Workflow

1. **Model selection** with PR-AUC (cross-validation).
2. **Set the threshold** using the use-case constraint (see Section 6).
3. **Standard binary evaluation** at this threshold (confusion matrix, precision, recall, F1).
4. **Subgroup analysis** across demographic variables.

## 6. Classification Strategy

For each person the model outputs a probability `p ∈ [0, 1]`. For binary classification **one** threshold is set:

`ŷ = 1` if `p ≥ T`, otherwise `ŷ = 0`.

### Choosing the threshold

Because our use case is a screening tool in which false negatives are much more costly than false positives (see the cost matrix in Section 3), we do not choose the threshold naively at 0.5 but in a **use-case-driven** way:

> *T = the smallest value at which a recall ≥ 0.80 is reached on the CV validation folds.*

Rationale: for a screening tool it is more important to identify as many truly affected people as possible (high recall) than to justify every alarm precisely. The specific recall target (0.7 / 0.8 / 0.9) is a team decision with justification.

**Important:** the threshold is chosen exclusively on the CV validation folds — never on the test set. The test set is used once and finally for evaluating the fully trained model. Any use of the test set for threshold choice, model selection or feature selection would create data leakage and bias the reported test performance.

### Output at the threshold (mandatory)

- Confusion matrix
- Accuracy (with a note on imbalance bias, as a context figure)
- Precision, recall, F1 (positive class)
- Precision, recall, F1 (negative class)
- Classification report (sklearn standard)
- Plus threshold-independent: PR-AUC, ROC-AUC

This fully documents the binary classifier and meets the standard expectation of a supervised-learning project.

## 7. Definition of Success

The model is usable as a binary classifier if:

1. **PR-AUC significantly above the no-skill baseline** (0.14). The exact attainable level is determined empirically, not fixed in advance.
2. **At the chosen threshold:** recall reaches the target defined in the recall constraint (e.g. ≥ 0.80); precision is reported transparently, not prescribed as a target.
3. **Subgroup performance comparable** across sex, age groups, income (no dramatic bias).
4. **Model interpretable** (feature importance / SHAP — which factors drive the prediction?).

## 8. Limitations

- **Self-report:** answers can be wrong, incomplete or biased.
- **Label noise:** "negatives" contain undiagnosed cases → performance ceiling.
- **Data age:** BRFSS 2015 → population, lifestyle and risk distributions may have shifted.
- **Geography:** U.S. + Puerto Rico → limited transferability to other countries.
- **Survey bias:** a telephone survey reaches certain groups less well (younger people, low income).
- **Coarse features:** no real biomarkers → a hard methodological ceiling.
- **Not a clinical diagnostic tool:** it only provides a recommendation to seek clarification; it does not replace a medical diagnosis.

## 9. Methodological Distinctiveness

- **Use-case-driven threshold choice** (recall constraint) instead of the default 0.5 — justified from the cost matrix.
- **Label-noise discussion:** the BRFSS label encodes diagnosis status, not disease status; the measured performance is therefore a lower bound on the true model quality.
- **Clean separation** of threshold-independent model selection (PR-AUC) from threshold-dependent binary classification.
- **Fairness / subgroup analysis** on a real survey dataset with demographic variables.
- **Deliberate metric choice:** accuracy discarded, PR-AUC primary, ROC-AUC only secondary — every decision justified by the class distribution.

## 10. Optional Extensions

The following points are **not mandatory components** of the binary-classification project. They can deepen the project methodologically if time and scope allow. Each is self-contained and can be addressed independently or omitted.

### 10.1 Calibration analysis (Brier score, calibration plot)

PR-AUC and ROC-AUC measure **discrimination** — how well the model separates positive from negative cases. They say nothing about whether the output probabilities are **realistic**. A prediction "p = 0.70" should actually mean that of 100 people with that score about 70 truly have diabetes.

**Method:**
- Calibration plot (reliability diagram): predicted probability against the actual fraction of positive cases; perfect = the diagonal.
- Brier score: mean squared error between prediction and the 0/1 label.
- If needed, recalibration with Platt scaling or isotonic regression (especially for Random Forest / XGBoost / SVM).

**Why optional:** for pure binary classification at a fixed threshold, calibration is not strictly required. It becomes relevant as soon as the continuous probability itself is communicated or used for tiered decisions (see 10.2).

### 10.2 Tiered deployment extension

Instead of a binary 0/1 output, the tool could give a **tiered recommendation** in three levels, based on two thresholds:

| Risk score (p) | Tier | Recommendation |
|---|---|---|
| `p < T_low` | Low | Maintain lifestyle, standard check-ups. |
| `T_low ≤ p < T_high` | Elevated | Adjust lifestyle, raise at the next doctor's visit. |
| `p ≥ T_high` | High | Timely HbA1c test with the GP. |

**Choice of `T_high`:** a balanced trade-off, e.g. recall ≥ 0.70 at precision ≥ 0.35 (team decision).

**Consistency:** identical model, identical binary training target — only an additional threshold interpretation of the continuous output. Requires sensible calibration (10.1).

**Why optional:** it extends the mandatory part (binary classification) with a deployment-oriented view but does not replace it.

### 10.3 False-positive profile analysis (deepening the label-noise argument)

From the label-noise insight in Section 4 follows a testable hypothesis: many of our model's false positives could in truth be **undiagnosed risk profiles**, not model errors.

**Method:**
- Extract the FP subgroup and compare its feature distribution with the TP subgroup.
- Expectation under a valid hypothesis: FPs resemble TPs in the classic risk factors (BMI, HighBP, older age, GenHlth) → a sign of undiagnosed diabetes status.
- The result supports the argument that measured performance is a **lower bound** on the true model quality.

**Why optional:** it deepens Section 4 empirically but is not required for the mandatory evaluation.

## Summary

We develop a binary classifier for diabetes risk estimation based on self-reported BRFSS lifestyle and health data. The target is `Diabetes_binary` (0/1); the classes are strongly imbalanced (~14 % positive). As the primary metric we use **PR-AUC**, because it is robust under imbalance and focused on the minority class; ROC-AUC serves only as a secondary metric for comparability with the literature. The classification threshold is chosen in a **use-case-driven** way via a recall constraint (screening logic, FN ≫ FP), not naively at 0.5. The standard output is the binary classification with full reporting (confusion matrix, precision/recall/F1, PR-AUC, ROC-AUC). The central methodological insight is that the BRFSS label encodes **diagnosis status**, not **disease status** — from which follow asymmetric label noise on the negative class and a performance lower bound that is contextualised honestly in the report. Possible methodological extensions (calibration, tiered deployment, FP profile analysis) are documented in Section 10 as optional.


## References

- Burrows, N. R., Hora, I., Geiss, L. S., Gregg, E. W., & Albright, A. (2017). Incidence of end-stage renal disease attributed to diabetes among persons with diagnosed diabetes — United States and Puerto Rico, 2000–2014. *MMWR. Morbidity and Mortality Weekly Report, 66*(43), 1165–1170. https://doi.org/10.15585/mmwr.mm6643a2
- Chicco, D., & Jurman, G. (2020). The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. *BMC Genomics, 21*, 6. https://doi.org/10.1186/s12864-019-6413-7
- Saito, T., & Rehmsmeier, M. (2015). The precision–recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432
- U.S. Centers for Disease Control and Prevention (CDC). *Behavioral Risk Factor Surveillance System (BRFSS) 2015 Codebook* (variable `DIABETE3`). https://www.cdc.gov/brfss/annual_data/2015/pdf/codebook15_llcp.pdf
- U.S. Centers for Disease Control and Prevention (CDC). *National Diabetes Statistics Report.* https://www.cdc.gov/diabetes/php/data-research/index.html
- National Center for Health Statistics (NCHS). *Prevalence of Diagnosed and Undiagnosed Diabetes …*, Data Brief 516 (November 2024). https://www.cdc.gov/nchs/products/databriefs/db516.htm